## Simulation 3b

In this simulation, we prepared the test data as follows:
We apply letter transposition to the target word without altering the order of the root letters (Transposed). After that, we substitute those letters (Substituted).

During training, we feed the model only the identity and position encodings of the target words, and train it to map each word to its semantic vector (300-dimensional).


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.layers import Dense, concatenate, GaussianNoise, Dropout
from tensorflow.keras.callbacks import Callback, EarlyStopping

In [ ]:
df_id   = pd.read_excel('Target1.xlsx', sheet_name='identity', index_col=0)
df_pos  = pd.read_excel('Target1.xlsx', sheet_name='position', index_col=0)
X_id    = df_id.values.astype(np.float32)
X_pos   = df_pos.values.astype(np.float32)

# training targets
df_sem = pd.read_excel('semantic_vec.xlsx', sheet_name='Sheet1', index_col=None)
Y_sem = df_sem.values[:477, :300].astype(np.float32)


inp_id    = Input(shape=(35,),  name='id_in')
tower_id  = Dense(700, activation='sigmoid')(inp_id)


inp_pos  = Input(shape=(100,), name='in_pos')
noisy_pos = GaussianNoise(stddev=0.2)(inp_pos)                # inject positional noise
tower_pos = Dense(700, activation='sigmoid')(noisy_pos)



merged    = concatenate([tower_id, tower_pos])
h         = Dense(500, activation='sigmoid')(merged)
out       = Dense(300, activation='sigmoid')(h)

model_3b = Model(inputs=[inp_id, inp_pos], outputs=out)


model_3b.compile(optimizer='adam', loss='mse')
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_3b.fit([X_id, X_pos], Y_sem, batch_size=25, epochs=700, callbacks=[es])
model_3b.save('simulation_3b_model.h5')

Epoch 1/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0984
Epoch 2/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0020
Epoch 3/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0018
Epoch 4/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0018
Epoch 5/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0018
Epoch 6/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0018
Epoch 7/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0018
Epoch 8/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0018
Epoch 9/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0017
Epoch 10/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0017
Epoch 11/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0017
Epoch 12/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0018
Epoch 13/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0018
Epoch 14/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0017
Epoch 15/700
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - lo

## Run the test sets

In [ ]:
df_sem = pd.read_excel('semantic_vec.xlsx', sheet_name='Sheet1', index_col=None)
Y_sem = df_sem.values[:477, :300].astype(np.float32)

data_lst= [ "Transposed", "Substituted"]
df_sim4 = pd.DataFrame()
for document in data_lst:
    X_id   = pd.read_excel(document+'.xlsx', sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos  = pd.read_excel(document+'.xlsx', sheet_name='position', index_col=0).values.astype(np.float32)

    pred = model_3b.predict([X_id, X_pos])

    valz = []
    for i in range(len(Y_sem)):
      r = np.corrcoef(pred[i], Y_sem[i])[0,1]
      valz.append(round(1-r,3))


    df_sim4[f'target-{document}']= valz

df_sim4.to_excel('simulation3b_correlations_test_7.xlsx')

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


# Correlation Distance Between words that share the same root

In [ ]:
import itertools

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model_3b.predict([X_id, X_pos])

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('targets with roots and semantics.xlsx', sheet_name='grouped by roots',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('Root'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(300)]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'Root': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('Root')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation3b_root_pairwise_distances_7.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_root_summary', index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words = pd.read_excel('Target1.xlsx', sheet_name='word', index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model_3b.predict([X_id, X_pos])


df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


PAIRS_XLSX  = 'targets with roots and semantics.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)

# keep exactly 4 columns in this order
out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation3b_4_conditions_pairs_with_distance_7.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
